In [ ]:

%pip install -q -U vllm==0.11.0 "transformers>=4.55.2,<5" "tokenizers>=0.21.1,<0.22"
%pip install -q sentence-transformers faiss-cpu rank_bm25 requests jedi

In [ ]:
import json
import os
import random
import numpy as np
import faiss
import requests
import re
import subprocess
import time
import threading
import torch
from datetime import datetime
from IPython.display import HTML, display

# 1. СТРОГАЯ ПРОВЕРКА GPU
if not torch.cuda.is_available():
    raise RuntimeError("🚨 ОШИБКА: GPU не найден! В верхнем меню Colab нажмите: 'Среда выполнения' -> 'Сменить среду выполнения' -> выберите 'T4 GPU'.")

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
last_logs = []

def watch(process) -> None:
    while True:
        line = process.stdout.readline()
        if not line and process.poll() is not None:
            break
        if line:
            clean_line = line.strip()
            last_logs.append(clean_line)
            if len(last_logs) > 15: last_logs.pop(0) # Храним последние 15 строк для дебага
            formatted_time = datetime.now().strftime("%H:%M:%S")
            display(HTML(f"<p style='color: cyan; margin: 0;'>VLLM [{formatted_time}]: {clean_line}</p>"))

print(f"🚀 Запуск сервера vLLM ({MODEL_NAME})...")

# 2. ОПТИМИЗИРОВАННЫЙ ЗАПУСК ДЛЯ COLAB T4
cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--host", "0.0.0.0",
    "--port", "9999",
    "--model", MODEL_NAME,
    "--max-model-len", "2048", # Снижено для гарантии запуска на 16GB VRAM
    "--dtype", "half",
    "--gpu-memory-utilization", "0.8", # Оставляем 20% памяти для моделей векторизации
    "--enforce-eager" # Отключаем графы CUDA (экономит память и ускоряет запуск)
]

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, # Направляем ошибки в общий лог
    text=True,
    bufsize=1
)

logs_watcher = threading.Thread(target=watch, args=(process, ))
logs_watcher.start()

server_ready = False
for _ in range(400): # Ждем до ~6.5 минут
    if not logs_watcher.is_alive():
        print("\n❌ ПРОЦЕСС VLLM УПАЛ! Последние логи:")
        for log in last_logs:
            print(f" > {log}")
        raise RuntimeError("Сервер vLLM аварийно завершил работу.")

    try:
        response = requests.get('http://localhost:9999/health')
        if response.status_code == 200:
            display(HTML("<h3 style='color: lime;'>✅ Сервер vLLM успешно запущен и готов к работе!</h3>"))
            server_ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not server_ready:
    raise RuntimeError("Не удалось дождаться запуска vLLM сервера по тайм-ауту.")

In [ ]:
os.makedirs("wsi_data", exist_ok=True)

def generate_mock_wsi_data(wsi_id: int) -> dict:
    return {
        "wsi_id": wsi_id,
        "wsi_class": f"Bethesda {random.randint(2, 6)}",
        "cell_characteristics": {
            "cellularity": random.randint(1000, 5000),
            "th_norm_cell_num": random.randint(500, 2000),
            "th_gurtle_cell_num": random.randint(0, 50),
            "lymphocyte_num": random.randint(10, 300),
            "mean_th_cell_area": round(random.uniform(30.0, 60.0), 2),
            "mean_th_cell_nuclear_cytoplasmic_ratio": round(random.uniform(0.1, 0.5), 2)
        },
        "cluster_characteristics": {
            "microfollicle_num": random.randint(0, 20),
            "mean_cluster_area": round(random.uniform(500.0, 1500.0), 2),
        }
    }

wsi_ids = [4, 5, 6]
generated_jsons = []
for wid in wsi_ids:
    data = generate_mock_wsi_data(wid)
    with open(f"wsi_data/{wid}.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    generated_jsons.append(data)

# Создаем бенчмарк на основе первого сгенерированного файла (ID 4)
wsi_4_data = generated_jsons[0]
benchmark_dataset = [
    {
        "query": "Какой общий класс системы Bethesda присвоен слайду номер 4?",
        "ground_truth_answer": f"Слайду ID 4 присвоен класс {wsi_4_data['wsi_class']}.",
        "expected_facts": [wsi_4_data['wsi_class'], "4"]
    },
    {
        "query": "Сколько клеток Гюртле обнаружено на слайде ID 4?",
        "ground_truth_answer": f"На слайде обнаружено {wsi_4_data['cell_characteristics']['th_gurtle_cell_num']} клеток Гюртле.",
        "expected_facts": [str(wsi_4_data['cell_characteristics']['th_gurtle_cell_num'])]
    },
    {
        "query": "Чему равен средний ядерно-цитоплазматический индекс на слайде 4?",
        "ground_truth_answer": f"Средний индекс равен {wsi_4_data['cell_characteristics']['mean_th_cell_nuclear_cytoplasmic_ratio']}.",
        "expected_facts": [str(wsi_4_data['cell_characteristics']['mean_th_cell_nuclear_cytoplasmic_ratio'])]
    }
]

with open("wsi_data/benchmark.json", "w", encoding="utf-8") as f:
    json.dump(benchmark_dataset, f, ensure_ascii=False, indent=4)

print("✅ JSON файлы и Benchmark успешно сгенерированы.")

In [ ]:
# Добавляем все нужные импорты прямо сюда для надежности
import json
import os
import requests
import numpy as np
import faiss
from typing import List, Dict, Tuple
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

# На всякий случай дублируем имя модели, если ячейка запускается отдельно
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

class LocalLLM:
    def __init__(self, model_name=MODEL_NAME, temperature=0.1):
        self.api_url = "http://localhost:9999/v1/chat/completions"
        self.model = model_name
        self.temperature = temperature

    def get_response(self, prompt: str, system_prompt: str = "") -> str:
        payload = {
            "model": self.model,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            "temperature": self.temperature,
            "max_tokens": 256
        }
        try:
            response = requests.post(self.api_url, json=payload)
            response.raise_for_status()
            return response.json()["choices"][0]["message"]["content"].strip()
        except Exception as e:
            print(f"Ошибка LLM: {e}")
            return ""

def json_to_medical_report(json_path: str) -> Tuple[str, dict]:
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    wsi_id = data.get("wsi_id", "Неизвестно")
    report = f"Медицинское заключение по гистологическому слайду (WSI) ID {wsi_id}.\n"
    report += f"Общий диагноз системы Bethesda: {data.get('wsi_class', 'Неизвестно')}.\n\n"
    cell_stats = data.get("cell_characteristics", {})
    report += f"Общая клеточность: {cell_stats.get('cellularity', 0)} клеток.\n"
    report += f"- Нормальные клетки ЩЖ: {cell_stats.get('th_norm_cell_num', 0)}\n"
    report += f"- Клетки Гюртле: {cell_stats.get('th_gurtle_cell_num', 0)}\n"
    report += f"- Лимфоциты: {cell_stats.get('lymphocyte_num', 0)}\n\n"
    report += f"Средняя площадь клетки: {cell_stats.get('mean_th_cell_area', 0)}.\n"
    report += f"Ядерно-цитоплазматический индекс: {cell_stats.get('mean_th_cell_nuclear_cytoplasmic_ratio', 0)}.\n\n"
    cluster_stats = data.get("cluster_characteristics", {})
    report += f"Микрофолликулярные структуры: {cluster_stats.get('microfollicle_num', 0)} шт.\n"
    report += f"Средняя площадь кластера: {cluster_stats.get('mean_cluster_area', 0)}."
    return report, {"wsi_id": wsi_id, "source": json_path}

class AdvancedRAGPipeline:
    def __init__(self, llm):
        self.llm = llm
        print("Загрузка моделей векторизации и реранжирования...")
        self.embedding_model = SentenceTransformer("cointegrated/rubert-tiny2")
        self.cross_encoder = CrossEncoder("DiTy/cross-encoder-russian-msmarco")
        self.chunks = []
        self.bm25 = None
        self.index = None

    def process_documents(self, documents: List[Tuple[str, dict]]):
        self.chunks = []
        for text, metadata in documents:
            paragraphs = [p.strip() for p in text.split('\n\n') if len(p.strip()) > 10]
            for p in paragraphs:
                self.chunks.append({"text": p, "metadata": metadata})
        print(f"✅ Векторная БД: подготовлено {len(self.chunks)} чанков (абзацев).")

        tokenized_corpus = [chunk["text"].lower().split() for chunk in self.chunks]
        self.bm25 = BM25Okapi(tokenized_corpus)
        embeddings = self.embedding_model.encode([c["text"] for c in self.chunks], show_progress_bar=False)
        self.index = faiss.IndexFlatL2(embeddings.shape[1])
        self.index.add(np.array(embeddings).astype('float32'))

    def search(self, query: str, top_k: int = 5, final_top_k: int = 2) -> List[Dict]:
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_indices = np.argsort(bm25_scores)[::-1][:top_k]

        query_embedding = self.embedding_model.encode([query]).astype('float32')
        _, faiss_indices = self.index.search(query_embedding, top_k)

        combined_indices = list(set(bm25_indices).union(set(faiss_indices[0])))
        candidate_chunks = [self.chunks[i] for i in combined_indices]

        cross_inp = [[query, chunk["text"]] for chunk in candidate_chunks]
        cross_scores = self.cross_encoder.predict(cross_inp)

        ranked_results = sorted(zip(candidate_chunks, cross_scores), key=lambda x: x[1], reverse=True)
        return [chunk for chunk, score in ranked_results[:final_top_k]]

    def answer_query(self, query: str) -> Tuple[str, List[Dict]]:
        retrieved = self.search(query)
        context = "\n---\n".join([f"[Слайд {doc['metadata']['wsi_id']}]: {doc['text']}" for doc in retrieved])
        sys_prompt = "Ты опытный врач-цитолог. Отвечай ТОЛЬКО на основе контекста. Будь краток и точен с цифрами."
        usr_prompt = f"Контекст:\n{context}\n\nВопрос: {query}\nОтвет:"
        return self.llm.get_response(usr_prompt, sys_prompt), retrieved

In [ ]:
class AdvancedRAGEvaluator:
    def __init__(self, rag_pipeline):
        self.rag = rag_pipeline

    def evaluate(self, eval_dataset: List[Dict]) -> Dict:
        metrics = {"hit_rate": 0, "context_relevance": 0, "faithfulness": 0, "answer_correctness": 0}
        total = len(eval_dataset)

        for item in eval_dataset:
            q, gt, facts = item["query"], item["ground_truth_answer"], item["expected_facts"]

            # Поиск
            retrieved = self.rag.search(q)
            context_used = "\n".join([c['text'] for c in retrieved])

            # Hit Rate
            if all(str(f).lower() in context_used.lower() for f in facts):
                metrics["hit_rate"] += 1

            # Context Relevance
            rel_prompt = f"Контекст:\n{context_used}\n\nВопрос: {q}\nСодержит ли контекст точный ответ на вопрос? Верни ТОЛЬКО цифру 1 (Да) или 0 (Нет)."
            if "1" in self.rag.llm.get_response(rel_prompt): metrics["context_relevance"] += 1

            # Генерация ответа и Faithfulness
            answer, _ = self.rag.answer_query(q)
            faith_prompt = f"Контекст:\n{context_used}\n\nОтвет: {answer}\nОснован ли ответ СТРОГО на контексте? Верни ТОЛЬКО 1 (Да) или 0 (Нет)."
            if "1" in self.rag.llm.get_response(faith_prompt): metrics["faithfulness"] += 1

            # Answer Correctness
            corr_prompt = f"Вопрос: {q}\nЭталон: {gt}\nГенерация: {answer}\nВерны ли фактические данные в генерации по сравнению с эталоном? Верни ТОЛЬКО 1 (Да) или 0 (Нет)."
            if "1" in self.rag.llm.get_response(corr_prompt): metrics["answer_correctness"] += 1

        return {k: round((v / total) * 100, 2) for k, v in metrics.items()}

In [ ]:
# 1. Читаем и текстуализируем JSON
json_files = [f"wsi_data/{f}" for f in os.listdir("wsi_data") if f.endswith(".json") and f != "benchmark.json"]
docs = [json_to_medical_report(f) for f in json_files]

# 2. Инициализируем систему
llm = LocalLLM()
rag = AdvancedRAGPipeline(llm=llm)
rag.process_documents(docs)

# 3. Тестовый запрос
test_q = "Сколько клеток Гюртле найдено на слайде 5?"
print(f"\nВопрос: {test_q}")
ans, sources = rag.answer_query(test_q)
print(f"Ответ RAG: {ans}")
print(f"Источник: Слайд ID {sources[0]['metadata']['wsi_id'] if sources else 'Не найден'}")

# 4. Автоматическая оценка
print("\n🔄 Запуск LLM-as-a-Judge Evaluation (Оценка по бенчмарку)...")
with open("wsi_data/benchmark.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

evaluator = AdvancedRAGEvaluator(rag)
results = evaluator.evaluate(eval_data)

print("\n=== 📊 ИТОГОВЫЕ МЕТРИКИ RAG (%) ===")
for metric, score in results.items():
    print(f"{metric.ljust(25)}: {score}%")

In [ ]:
import requests

EVAL_SERVICE_URL = "https://a1b2-c3d4.ngrok-free.app/evaluate" # Ваш URL от Ngrok

def evaluate_rag_generation(query, answer, contexts, ground_truth=None):
    payload = {
        "query": query,
        "answer": answer,
        "contexts": contexts,
        "ground_truth": ground_truth
    }
    
    try:
        response = requests.post(EVAL_SERVICE_URL, json=payload)
        response.raise_for_status()
        metrics = response.json()
        print(f"📊 Метрики оценки: {metrics}")
        return metrics
    except requests.exceptions.RequestException as e:
        print(f"❌ Ошибка связи с сервисом оценки: {e}")
        return None

# --- Как это выглядит в вашем рабом процессе в Colab ---
test_q = "Определи категорию по описанию: П 29 (№3683/20). На фоне жидкого коллоида..."
ans, sources = rag.answer_query(test_q)
context_texts = [doc['text'] for doc in sources]

# Отправляем в микросервис!
metrics = evaluate_rag_generation(
    query=test_q,
    answer=ans,
    contexts=context_texts,
    ground_truth="Неинформативное исследование Bethesda - I" # (Если есть)
)

In [ ]:
%pip install -q -U ragas datasets langchain langchain-openai langchain-huggingface

In [ ]:


import json
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)
from ragas.run_config import RunConfig
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

print("⚙️ Настройка Ragas для работы с локальным vLLM...")

# 2. Оборачиваем наш локальный vLLM сервер в интерфейс Langchain для Ragas
# API ключ не нужен, но Langchain требует передачи любой строки.
ragas_llm = ChatOpenAI(
    model=MODEL_NAME,
    openai_api_key="EMPTY",
    openai_api_base="http://localhost:9999/v1",
    max_tokens=512,
    temperature=0.1,
)

# Оборачиваем модель эмбеддингов для расчетов (например, для answer_relevancy)
ragas_embeddings = HuggingFaceEmbeddings(
    model_name="cointegrated/rubert-tiny2"
)

# 3. Загружаем эталонный датасет Bethesda
with open("bethesda_ground_truth.json", "r", encoding="utf-8") as f:
    bethesda_data = json.load(f)

# 4. Индексируем цитограммы в нашу векторную БД (симулируем базу знаний клиники)
print("📚 Загрузка цитограмм в векторную базу данных...")
bethesda_docs = []
for item in bethesda_data:
    # Сохраняем весь simulated_context как документ в базе
    doc_text = item["simulated_context"]
    metadata = {"wsi_id": f"Patient_{item['id']}", "source": "bethesda_json"}
    bethesda_docs.append((doc_text, metadata))

# Обновляем БД в RAG
rag.process_documents(bethesda_docs)

# 5. Собираем предсказания (Прогоняем датасет через наш RAG)
print("🤖 Генерация ответов через RAG для датасета...")

data_for_ragas = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

for item in bethesda_data:
    # ХАК: Делаем запрос уникальным, чтобы RAG мог найти именно этот контекст в БД.
    # Берем первые ~30 символов (например "П 29 (№3683/20)") как идентификатор.
    patient_id = item["simulated_context"][:30].strip()
    smart_query = f"Определи диагностическую категорию Bethesda для: {patient_id}..."

    # Получаем ответ и найденные контексты от нашего RAG
    answer, sources = rag.answer_query(smart_query)

    # Формируем список найденных текстов
    retrieved_contexts = [doc["text"] for doc in sources]

    # Заполняем словари для Ragas
    data_for_ragas["question"].append(smart_query)
    data_for_ragas["answer"].append(answer)
    data_for_ragas["contexts"].append(retrieved_contexts)
    data_for_ragas["ground_truth"].append(item["ground_truth"])

# Конвертируем в HuggingFace Dataset
ragas_dataset = Dataset.from_dict(data_for_ragas)

# 6. Запускаем оценку Ragas
print("\n📊 Запуск оценки Ragas (Context Precision, Recall, Faithfulness, Relevancy)...")
print("⏳ Это может занять несколько минут, LLM выступает в роли судьи...")

# Ограничиваем количество потоков, чтобы не перегрузить Colab T4 (max_workers=2 или 1)
run_config = RunConfig(max_workers=2, timeout=120)

result = evaluate(
    dataset=ragas_dataset,
    metrics=[
        context_precision, # Были ли нужные куски текста в топе выдачи?
        context_recall,    # Нашел ли ретривер всё необходимое для ответа?
        faithfulness,      # Не выдумал ли LLM факты, которых нет в контексте?
        answer_relevancy   # Насколько ответ отвечает на заданный вопрос?
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=run_config,
    raise_exceptions=False # Чтобы не падал весь процесс, если LLM выдаст кривой JSON
)


In [ ]:
import pandas as pd 
print("\n" + "="*40)
print("🏆 ИТОГОВЫЕ МЕТРИКИ RAGAS")
print("="*40)

# Конвертируем результаты в Pandas DataFrame (это самый стабильный метод)
df_results = result.to_pandas()

# Названия колонок с метриками, которые мы считали
metrics_used = ["context_precision", "context_recall", "faithfulness", "answer_relevancy"]

for metric in metrics_used:
    if metric in df_results.columns:
        # Считаем среднее значение по столбцу для всех вопросов
        mean_score = df_results[metric].mean()
        
        # Защита от NaN (если LLM не смогла сгенерировать правильный JSON для оценки)
        if pd.isna(mean_score):
            print(f"🔹 {metric.ljust(20)} : Ошибка оценки (NaN)")
        else:
            print(f"🔹 {metric.ljust(20)} : {round(mean_score * 100, 2)}%")

print("\nДетальные результаты по каждому примеру:")

# Собираем колонки динамически, проверяя, как Ragas их назвал в этой версии
display_cols = []

# Ищем колонку с вопросом
if "user_input" in df_results.columns: display_cols.append("user_input")
elif "question" in df_results.columns: display_cols.append("question")

# Ищем колонку с ответом
if "response" in df_results.columns: display_cols.append("response")
elif "answer" in df_results.columns: display_cols.append("answer")

# Добавляем нужные метрики
for m in ["faithfulness", "context_precision", "context_recall"]:
    if m in df_results.columns:
        display_cols.append(m)

# Если почему-то ни одна текстовая колонка не нашлась, выводим все
if not display_cols:
    display_cols = df_results.columns
